# AXE 2 — 02. ALS Model (Truth Edition 2.0)
## 🎯 Objectif : Réduire le biais de fréquence et forcer tes favoris réels.

In [1]:
import os, re, numpy as np, pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS
from difflib import SequenceMatcher

spark = SparkSession.builder \
    .appName("MyDigitalTwin-ALS-Truth") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
WAREHOUSE = "warehouse"

26/04/06 19:15:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
def clean_title(t):
    t = t.split(',')[0].split('(')[0].split(':')[0]
    t = re.sub(r'[^\w\s]', '', t)
    return t.strip().lower()

def find_top_md():
    for root, dirs, files in os.walk('.'):
        if 'top.md' in files: return os.path.join(root, 'top.md')
    for root, dirs, files in os.walk('..'):
        if 'top.md' in files: return os.path.join(root, 'top.md')
    return None

TOP_MD_PATH = find_top_md()
netflix_favs = []
if TOP_MD_PATH:
    with open(TOP_MD_PATH, 'r', encoding='utf-8') as f:
        for l in f.readlines():
            l = l.strip()
            if not l or l.startswith('##'): continue
            c = clean_title(l)
            if len(c) > 2: netflix_favs.append(c)

spotify_artists = ["Damso", "Tiakola", "Travis Scott", "Bad Bunny", "Ninho", "Metro Boomin", "Ziak", "Gazo", "Freeze Corleone"]
spotify_artists_lower = [a.lower() for a in spotify_artists]
print(f"✅ {len(netflix_favs)} favoris Netflix et {len(spotify_artists)} artistes Spotify chargés.")

✅ 90 favoris Netflix et 9 artistes Spotify chargés.


In [3]:
def run_engine(platform_name):
    print(f"\n🚀 Engine: {platform_name.upper()}")
    
    # 1. Chargement et Nettoyage de type (Evite Py4JJavaError)
    df = spark.read.parquet(os.path.join(WAREHOUSE, "interactions")).filter(F.col("platform") == platform_name)
    df = df.filter(F.col("user_id").isNotNull() & F.col("item_id").isNotNull())
    df = df.withColumn("user_id", F.col("user_id").cast("int"))
    df = df.withColumn("item_id", F.col("item_id").cast("int"))
    
    # --- CORRECTION 1 : LOG SCALING ---
    df = df.withColumn("weight", F.log1p(F.col("play_count").cast("double")))
    
    if df.count() == 0: return pd.DataFrame()

    # --- CORRECTION 2 : ALS STABLE ---
    als = ALS(userCol="user_id", itemCol="item_id", ratingCol="weight", rank=20, maxIter=20, regParam=0.15, implicitPrefs=True, seed=42)
    model = als.fit(df)
    
    factors = model.itemFactors.toPandas()
    f_matrix = np.vstack(factors["features"].values)
    f_norm = f_matrix / (np.linalg.norm(f_matrix, axis=1, keepdims=True) + 1e-8)
    
    totals = df.groupBy("item_id", "item_title").agg(F.sum("play_count").alias("plays")).toPandas()
    
    # --- CORRECTION 3 : FUZZY MATCHING POUR LES ANCRES ---
    def get_match_score(t, fav_list):
        ct = clean_title(t)
        for f in fav_list:
            if f in ct or ct in f: return 1.0
            ratio = SequenceMatcher(None, ct, f).ratio()
            if ratio > 0.85: return 1.0
        return 0.0

    if platform_name == "netflix":
        anchors = totals[totals["item_title"].apply(lambda x: get_match_score(x, netflix_favs) == 1.0)]
    else:
        anchors = totals[totals["item_title"].str.lower().str.contains('|'.join(spotify_artists_lower))]
    
    print(f"📌 Profil basé sur {len(anchors)} ancres détectées.")
    
    id_to_idx = {int(id_): idx for idx, id_ in enumerate(factors["id"].values)}
    indices = [id_to_idx[int(i)] for i in anchors["item_id"].values if int(i) in id_to_idx]
    
    ref = f_norm[indices].mean(axis=0)
    ref /= (np.linalg.norm(ref) + 1e-8)
    
    totals["score"] = f_norm @ ref
    
    # --- CORRECTION 4 : BOOST EXPLICITE ---
    def apply_expert_boost(row):
        boost = 1.3
        if platform_name == "netflix":
            if get_match_score(row["item_title"], netflix_favs) == 1.0: boost += 0.5
        else:
            if any(art in row["item_title"].lower() for art in spotify_artists_lower): boost += 0.5
        return boost

    totals["score"] *= totals.apply(apply_expert_boost, axis=1)
    
    # Normalisation 0-100
    s_min, s_max = totals["score"].min(), totals["score"].max()
    totals["score"] = ((totals["score"] - s_min) / (s_max - s_min) * 100).round(1)
    totals["platform"] = platform_name
    return totals

In [4]:
try:
    n_res = run_engine("netflix")
    s_res = run_engine("spotify")
    print("\n--- VERIF NETFLIX ---")
    print(n_res.sort_values("score", ascending=False).head(15)[["item_title", "score"]].to_string(index=False))
    print("\n--- VERIF SPOTIFY ---")
    print(s_res.sort_values("score", ascending=False).head(15)[["item_title", "score"]].to_string(index=False))
except Exception as e:
    print(f"❌ Erreur fatale : {e}")


🚀 Engine: NETFLIX


📌 Profil basé sur 26 ancres détectées.

🚀 Engine: SPOTIFY


📌 Profil basé sur 368 ancres détectées.

--- VERIF NETFLIX ---
                    item_title  score
                    Archive 81  100.0
          Love, Death & Robots   99.3
               Stranger Things   97.7
                Inventing Anna   96.0
               BoJack Horseman   92.7
                       En bref   91.1
                   Break Point   88.5
           Alice in Borderland   86.1
                        Arcane   85.1
                    Baki Hanma   82.8
                         Joker   81.9
Le Monde incroyable de Gumball   81.7
                    The 8 Show   81.1
                  Prison Break   80.8
                 The Gentlemen   79.5

--- VERIF SPOTIFY ---
                                           item_title  score
                            Freeze corleone — Shavkat  100.0
                                Ziak — Pistol & Zamal   94.5
                                    Ninho — STOCKHOLM   92.3
                                           Gazo — Job   92.1
 

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
final_df = pd.concat([n_res, s_res])
final_df["rank"] = final_df.groupby("platform")["score"].rank(ascending=False, method="first").astype(int)
data_list = []
for _, r in final_df.iterrows():
    data_list.append({"item_id": int(r["item_id"]), "item_title": str(r["item_title"]), "platform": str(r["platform"]), "play_count": int(r["plays"]), "predicted_score": float(r["score"]), "rank": int(r["rank"])})
schema = StructType([StructField("item_id", IntegerType(), True), StructField("item_title", StringType(), True), StructField("platform", StringType(), True), StructField("play_count", IntegerType(), True), StructField("predicted_score", FloatType(), True), StructField("rank", IntegerType(), True)])
spark.createDataFrame(data_list, schema=schema).write.mode("overwrite").parquet(os.path.join(WAREHOUSE, "als_scores"))
print("✅ Scores sauvegardés !")
spark.stop()